# HODGE v10a.31 — M5-ONLY Blind \(m_5/m_6/m_7\) A100 Runner

This notebook treats **no M4** literally.

It does not execute the v10a.26 notebook. It extracts only the dependency-closed definitions required by the order-generic engine, and it rejects the bootstrap if any fourth-order production state is selected.

It patches v10a.28 before execution so that:

- order 4 is not an allowed runtime order;
- the all-order \(O4\)–\(O7\) synthetic sweep becomes a requested-order-only test;
- the v10a.26 one-face \(O4\) comparator is removed;
- the order-4 support-census regression is removed;
- the hard-coded lower-order target table, including the \(m_4\) target gate, is removed;
- the order-5 one-face smoke is reused as the matching M5 production shape;
- firewall, M5 census, and M5 production execute once in a single order-5 invocation.

The first scientific calculation is therefore M5. No published \(m_5,m_6,m_7\) targets are present.


In [ ]:
from __future__ import annotations

import ast
import builtins
import hashlib
import json
import os
import platform
import shutil
import subprocess
import symtable
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import nbformat
import numpy as np
import opt_einsum as oe
import sympy as sp

# ------------------------------ USER SETTINGS ------------------------------
USE_GOOGLE_DRIVE = True
WORKDIR_NAME = "HODGE_BLIND_M5_M7_M5_ONLY"
AUTO_UPLOAD_MISSING_FILES = True

V26_NAME = "NB_O4_hodge_v10a26_factor52complete_exactsw_rootedoracle_a100.ipynb"
V28_NAME = "ENGINE_O4_hodge_v10a28_orderaware_gram_firewall_a100.py"

# A100 production: zero means unlimited in this invocation.
M5_MAX_NEW_SHAPES = 0
M5_TIME_BUDGET_MINUTES = 0
GPU_SW_MIN_DIM = 64
HERM_AUDIT_PAIRS = 24
DUPLICATE_CHECKS = 1
HEARTBEAT_SECONDS = 20

# Higher orders use the same no-M4 engine. This does not force an external
# validation flag; each order remains target-blind and writes a separate summary.
HIGHER_ORDER_MAX_NEW_SHAPES = 1
HIGHER_ORDER_TIME_BUDGET_MINUTES = 30

print("UTC:", datetime.now(timezone.utc).isoformat())
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Execution policy: requested orders 5, 6, 7 only; no order-4 execution or target gate.")


In [ ]:
# Safe A100/CUDA check. Do not replace an existing CuPy installation blindly.
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        check=True, capture_output=True, text=True,
    )
    print("nvidia-smi:", smi.stdout.strip())
except Exception as exc:
    raise RuntimeError("No usable NVIDIA runtime detected. Select an A100 runtime.") from exc

try:
    import cupy as cp
except Exception as exc:
    raise RuntimeError(
        "CuPy is not importable. Use an A100 runtime with a compatible CuPy/CUDA build; "
        "this notebook will not install an arbitrary wheel over the live runtime."
    ) from exc

GPU_COUNT = int(cp.cuda.runtime.getDeviceCount())
if GPU_COUNT < 1:
    raise RuntimeError("No CUDA device is visible to CuPy.")

props = cp.cuda.runtime.getDeviceProperties(0)
GPU_NAME = props["name"].decode() if isinstance(props["name"], (bytes, bytearray)) else str(props["name"])
free_b, total_b = cp.cuda.runtime.memGetInfo()
print("CUDA devices:", GPU_COUNT)
print("GPU 0:", GPU_NAME)
print(f"GPU memory: free={free_b/2**30:.2f} GiB / total={total_b/2**30:.2f} GiB")
if "A100" not in GPU_NAME.upper():
    print("WARNING: the runtime is not reporting an A100; execution is still permitted.")


In [ ]:
# Durable checkpoints on Drive when available.
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        WORKDIR = Path("/content/drive/MyDrive") / WORKDIR_NAME
    except Exception as exc:
        print("Drive mount unavailable; using /content:", exc)
        WORKDIR = Path("/content") / WORKDIR_NAME
else:
    WORKDIR = Path("/content") / WORKDIR_NAME

WORKDIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR = WORKDIR / "generated_sources"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
print("WORKDIR:", WORKDIR)


In [ ]:
# Locate or upload the two source artifacts.
CONTENT = Path("/content")

def locate(name: str) -> Path | None:
    for path in (CONTENT / name, WORKDIR / name, Path.cwd() / name):
        if path.exists():
            return path
    return None


def upload_if_missing(name: str) -> Path:
    path = locate(name)
    if path is not None:
        return path
    if not AUTO_UPLOAD_MISSING_FILES:
        raise FileNotFoundError(name)
    try:
        from google.colab import files
    except Exception as exc:
        raise FileNotFoundError(f"{name} not found and Colab upload is unavailable") from exc
    print(f"Upload: {name}")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f"No file uploaded for {name}")
    if name in uploaded:
        return CONTENT / name
    if len(uploaded) != 1:
        raise RuntimeError(f"Expected {name}; received {list(uploaded)}")
    source_name = next(iter(uploaded))
    source = CONTENT / source_name
    destination = CONTENT / name
    if source != destination:
        shutil.move(str(source), str(destination))
    return destination


V26_PATH = upload_if_missing(V26_NAME)
V28_PATH = upload_if_missing(V28_NAME)
print("v10a.26:", V26_PATH)
print("v10a.28:", V28_PATH)


In [ ]:
# Provenance hashes. No requested-order coefficient target is loaded.
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


V26_SHA256 = sha256_file(V26_PATH)
V28_SHA256 = sha256_file(V28_PATH)
print("v10a.26 SHA-256:", V26_SHA256)
print("v10a.28 SHA-256:", V28_SHA256)

V28_ORIGINAL_SOURCE = V28_PATH.read_text(encoding="utf-8", errors="strict")
if "target coefficient           : NOT LOADED" not in V28_ORIGINAL_SOURCE:
    raise RuntimeError("The blind-target declaration is absent from the supplied v10a.28 source.")
print("Blind-source declaration found.")


## Definitions-only v10a.26 bootstrap

This does not execute the notebook. It parses the source, takes the transitive dependency closure of the names needed by v10a.28, writes a new definitions library, and executes only that generated library.

The bootstrap fails closed if it selects `shape_cache`, the fourth-order operator objects, the finite-cluster result, or any known M4 production call.


In [ ]:
V26_BOOTSTRAP_ROOTS = (
    "L", "N", "faces", "verts", "T1_POLS", "anchor_faces", "V23C_ROOT",
    "V23C_POL", "_FAST_EPS", "oe", "LXState", "_V17_VAC",
    "_v17_apply_W_faces", "_v17_apply_W_labeled", "_v17_connected",
    "_v17_phys_index", "_v17_translate_support", "_v17_translate_face",
    "_v23c_split_h0", "_v23c_rooted_connected_subsets", "_v24c_shape_key",
    "_v24c_candidate_supports", "_v10a3_face_state",
    "_v10a3_physical_blocks", "_v10a3_compress_state", "_v9_flux_key_state",
    "_joint_canon_states", "lx_combine_bra_ket",
    "_v23_sw_exact", "_v23_sp", "_v23_random", "_V23CF",
    "_v26_singlet_multiplicity", "_V17_NEIGH",
)
V26_BOOTSTRAP_SEEDS = ()

_MUTATING_METHODS = {
    "add", "append", "clear", "discard", "extend", "insert", "pop",
    "remove", "setdefault", "sort", "update",
}
FORBIDDEN_BOOTSTRAP_NAMES = {
    "shape_cache", "M4_ORACLE", "V26_RESULT", "Dop", "K4op", "C_COLD", "VERDICT",
}
FORBIDDEN_BOOTSTRAP_CALLS = {
    "_v26_load_checkpoint", "_v23c_fit_cluster",
}


def _base_name(node):
    while isinstance(node, (ast.Attribute, ast.Subscript)):
        node = node.value
    return node.id if isinstance(node, ast.Name) else None


def _defined_or_mutated_names(stmt):
    names = set()

    class Visitor(ast.NodeVisitor):
        def visit_FunctionDef(self, node):
            names.add(node.name)
            # Function-body stores are local/module-runtime behavior, not definitions here.

        visit_AsyncFunctionDef = visit_FunctionDef

        def visit_ClassDef(self, node):
            names.add(node.name)

        def visit_Import(self, node):
            for alias in node.names:
                names.add(alias.asname or alias.name.split(".")[0])

        def visit_ImportFrom(self, node):
            for alias in node.names:
                if alias.name != "*":
                    names.add(alias.asname or alias.name)

        def visit_Name(self, node):
            if isinstance(node.ctx, (ast.Store, ast.Del)):
                names.add(node.id)

        def visit_Subscript(self, node):
            if isinstance(node.ctx, (ast.Store, ast.Del)):
                base = _base_name(node)
                if base:
                    names.add(base)
            self.generic_visit(node)

        def visit_Attribute(self, node):
            if isinstance(node.ctx, (ast.Store, ast.Del)):
                base = _base_name(node)
                if base:
                    names.add(base)
            self.generic_visit(node)

        def visit_Call(self, node):
            if isinstance(node.func, ast.Attribute) and node.func.attr in _MUTATING_METHODS:
                base = _base_name(node.func.value)
                if base:
                    names.add(base)
            self.generic_visit(node)

    Visitor().visit(stmt)
    return names


def _statement_dependencies(stmt, all_defined_names):
    text = ast.unparse(stmt)
    table = symtable.symtable(text, "<v26-bootstrap-statement>", "exec")
    dependencies = set()

    def walk(tab):
        module_scope = tab.get_type() == "module"
        for symbol in tab.get_symbols():
            if not symbol.is_referenced():
                continue
            if module_scope or symbol.is_global():
                dependencies.add(symbol.get_name())
        for child in tab.get_children():
            walk(child)

    walk(table)
    return dependencies & all_defined_names


def _called_names(stmt):
    names = set()
    for node in ast.walk(stmt):
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                names.add(node.func.id)
            elif isinstance(node.func, ast.Attribute):
                names.add(node.func.attr)
    return names


def build_v26_bootstrap(notebook_path: Path, output_path: Path, manifest_path: Path):
    document = nbformat.read(notebook_path, as_version=4)
    source = "\n\n".join(
        cell.source for cell in document.cells
        if cell.cell_type == "code" and cell.source.strip()
    )
    tree = ast.parse(source, filename=str(notebook_path))
    statements = list(tree.body)

    defines = []
    definition_map = {}
    for index, statement in enumerate(statements):
        names = _defined_or_mutated_names(statement)
        defines.append(names)
        for name in names:
            definition_map.setdefault(name, set()).add(index)

    all_names = set(definition_map)
    selected = {
        index for index, statement in enumerate(statements)
        if isinstance(statement, (ast.Import, ast.ImportFrom))
    }
    pending = list(dict.fromkeys((*V26_BOOTSTRAP_ROOTS, *V26_BOOTSTRAP_SEEDS)))
    resolved = set()

    while pending:
        name = pending.pop()
        if name in resolved:
            continue
        resolved.add(name)
        providers = sorted(definition_map.get(name, ()))
        if not providers and name in V26_BOOTSTRAP_ROOTS and not hasattr(builtins, name):
            raise RuntimeError(f"Definitions-only bootstrap cannot resolve required symbol: {name}")
        for index in providers:
            if index not in selected:
                selected.add(index)
                pending.extend(_statement_dependencies(statements[index], all_names) - resolved)

    selected_names = set().union(*(defines[index] for index in selected)) if selected else set()
    forbidden_names = sorted(selected_names & FORBIDDEN_BOOTSTRAP_NAMES)
    if forbidden_names:
        raise RuntimeError(
            "Definitions-only bootstrap selected forbidden M4 production state: "
            + ", ".join(forbidden_names)
        )

    forbidden_calls = []
    for index in sorted(selected):
        statement = statements[index]
        # Function definitions may mention old helpers without executing them; the roots above
        # deliberately exclude those functions. Only selected top-level runtime calls are blocked.
        if isinstance(statement, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            continue
        hits = _called_names(statement) & FORBIDDEN_BOOTSTRAP_CALLS
        if hits:
            forbidden_calls.append((index, sorted(hits)))
    if forbidden_calls:
        raise RuntimeError(f"Definitions-only bootstrap selected forbidden runtime calls: {forbidden_calls}")

    module = ast.Module(body=[statements[index] for index in sorted(selected)], type_ignores=[])
    ast.fix_missing_locations(module)
    bootstrap_source = ast.unparse(module) + "\n"
    compile(bootstrap_source, str(output_path), "exec")
    output_path.write_text(bootstrap_source, encoding="utf-8")

    manifest = {
        "schema": "hodge-v10a31-v26-definitions-only-bootstrap-v1",
        "source_sha256": sha256_file(notebook_path),
        "bootstrap_sha256": sha256_file(output_path),
        "roots": list(V26_BOOTSTRAP_ROOTS),
        "seeds": list(V26_BOOTSTRAP_SEEDS),
        "source_statement_count": len(statements),
        "selected_statement_count": len(selected),
        "selected_statement_indices": sorted(selected),
        "selected_defined_names": sorted(selected_names),
        "forbidden_names_selected": forbidden_names,
        "forbidden_runtime_calls_selected": forbidden_calls,
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


BOOTSTRAP_PATH = SOURCE_DIR / f"v26_definitions_only_{V26_SHA256[:12]}.py"
BOOTSTRAP_MANIFEST = SOURCE_DIR / f"v26_definitions_only_{V26_SHA256[:12]}.json"
rebuild = True
if BOOTSTRAP_PATH.exists() and BOOTSTRAP_MANIFEST.exists():
    try:
        prior = json.loads(BOOTSTRAP_MANIFEST.read_text(encoding="utf-8"))
        rebuild = any((
            prior.get("source_sha256") != V26_SHA256,
            prior.get("roots") != list(V26_BOOTSTRAP_ROOTS),
            prior.get("seeds") != list(V26_BOOTSTRAP_SEEDS),
            sha256_file(BOOTSTRAP_PATH) != prior.get("bootstrap_sha256"),
        ))
    except Exception:
        rebuild = True
if rebuild:
    BOOTSTRAP_INFO = build_v26_bootstrap(V26_PATH, BOOTSTRAP_PATH, BOOTSTRAP_MANIFEST)
else:
    BOOTSTRAP_INFO = prior

print(
    "Bootstrap selected statements:",
    BOOTSTRAP_INFO["selected_statement_count"], "/", BOOTSTRAP_INFO["source_statement_count"],
)
print("Bootstrap:", BOOTSTRAP_PATH)
print("Bootstrap SHA-256:", sha256_file(BOOTSTRAP_PATH))

# Safe initializer configuration. These do not launch v10a.26 production.
os.environ["PREFER_GPU"] = "1"
os.environ["GLUE_L"] = "5"
os.environ["V10A23_CLUSTER_POL"] = "2"
os.environ["V10A23_CLUSTER_PROGRESS"] = "0"
os.environ["V10A7_SUPPORT_POLS"] = "2"
os.environ["V10A7_RECHECK_Q1"] = "0"
os.environ["V10A7_UNBLIND"] = "0"

bootstrap_source = BOOTSTRAP_PATH.read_text(encoding="utf-8")
exec(compile(bootstrap_source, str(BOOTSTRAP_PATH), "exec"), globals())

missing = [name for name in V26_BOOTSTRAP_ROOTS if name not in globals()]
if missing:
    raise RuntimeError("Definitions-only bootstrap missed required symbols: " + ", ".join(missing))

for forbidden_name in FORBIDDEN_BOOTSTRAP_NAMES:
    if forbidden_name in globals():
        raise RuntimeError(f"Forbidden M4 production state leaked into bootstrap: {forbidden_name}")

print("v10a.26 notebook execution: SKIPPED")
print("v10a.26 definitions-only namespace: PASS")
print("M4 production objects present: NONE")
print("L =", L, "N =", N, "faces =", len(faces))


## Patch v10a.28 into a requested-order-only engine

The patch is anchored to the supplied source and fails if the expected source layout differs. There is no fallback to the unpatched file.


In [ ]:
def _replace_once(source: str, old: str, new: str, label: str) -> str:
    count = source.count(old)
    if count != 1:
        raise RuntimeError(f"Patch anchor {label!r} expected once, found {count}")
    return source.replace(old, new, 1)


def patch_v28_m5_only(source: str) -> str:
    original = source

    # New checkpoint schema: old runner checkpoints cannot be mixed into this run.
    source = _replace_once(
        source,
        'V28_SCHEMA = "hodge-v10a28-order-aware-krylov-gram-haar9-v1"',
        'V28_SCHEMA = "hodge-v10a31-m5-only-krylov-gram-haar9-v1"',
        "schema",
    )

    # Default and legal runtime orders: 5, 6, 7 only.
    source = _replace_once(
        source,
        'V28_ORDER = int(os.environ.get("V28_ORDER", "4"))',
        'V28_ORDER = int(os.environ.get("V28_ORDER", "5"))',
        "default order",
    )
    source = _replace_once(
        source,
        'if V28_ORDER not in (4, 5, 6, 7):\n    raise ValueError("V28_ORDER must be 4, 5, 6, or 7")',
        'if V28_ORDER not in (5, 6, 7):\n    raise ValueError("M5-only runner permits V28_ORDER 5, 6, or 7")',
        "allowed orders",
    )

    # Remove globals that were needed only by the old O4 comparator path.
    source = _replace_once(
        source,
        '    "_v24c_candidate_supports", "_v10a3_face_state", "_v10a3_h0_state_inner",\n',
        '    "_v24c_candidate_supports", "_v10a3_face_state",\n',
        "required globals h0 comparator",
    )
    source = _replace_once(
        source,
        '    "_joint_canon_states", "lx_combine_bra_ket", "_v26_sw_blocks",\n',
        '    "_joint_canon_states", "lx_combine_bra_ket",\n',
        "required globals old sw",
    )
    source = _replace_once(
        source,
        '    "_v26_singlet_multiplicity", "_V17_NEIGH", "_v23c_fit_cluster",\n',
        '    "_v26_singlet_multiplicity", "_V17_NEIGH",\n',
        "required globals old fit",
    )

    # The synthetic Krylov theorem regression runs only at the requested order.
    source = _replace_once(
        source,
        '    for order in (4, 5, 6, 7):\n',
        '    for order in (V28_ORDER,):\n',
        "all-order synthetic loop",
    )

    # Replace the old generic/O4-comparator regression section with current-order tests.
    regression_start = source.index('print("\\n[2] ORDER-GENERIC SW REGRESSION")')
    regression_end_marker = '# ---------------------------------------------------------------------------\n# 4. Generic bidirectional support history census'
    regression_end = source.index(regression_end_marker, regression_start)
    current_order_block = r'''
print(f"\n[2] ORDER-{V28_ORDER} SW/BCH REGRESSION")
V28_SW_REGRESSION_ERROR = _v28_sw_regression(V28_ORDER)
v28_gate(
    f"NumPy SW recursion matches exact rational BCH through O(u^{V28_ORDER})",
    V28_SW_REGRESSION_ERROR < V28_SW_TOL,
    f"max error={V28_SW_REGRESSION_ERROR:.3e}",
)
V28_GPU_SW_REGRESSION_ERROR = _v28_gpu_sw_regression(V28_ORDER)
v28_gate(
    f"order-{V28_ORDER} GPU and CPU SW/BCH backends agree",
    V28_GPU_SW_REGRESSION_ERROR < 2e-9,
    (f"max error={V28_GPU_SW_REGRESSION_ERROR:.3e}" if V28_GPU_ENABLED else "GPU unavailable; CPU fallback"),
)
V28_ORDER_BAND_ERRORS, V28_ODD_DEEP_SENSITIVITY = _v28_order_band_regression()
v28_gate(
    f"order-{V28_ORDER} Krylov truncation matches the full band model",
    V28_ORDER_BAND_ERRORS[V28_ORDER] < V28_SW_TOL,
    f"O{V28_ORDER}={V28_ORDER_BAND_ERRORS[V28_ORDER]:.2e}",
)
if V28_ORDER % 2:
    v28_gate(
        f"order-{V28_ORDER} regression detects removal of the deepest self-block",
        V28_ODD_DEEP_SENSITIVITY[V28_ORDER] > 1e-8,
        f"response={V28_ODD_DEEP_SENSITIVITY[V28_ORDER]:.2e}",
    )
else:
    v28_gate(
        f"order-{V28_ORDER} uses the even-order deepest-self omission theorem",
        V28_ORDER_BAND_ERRORS[V28_ORDER] < V28_SW_TOL,
        "no odd-order Q_d W Q_d block is required",
    )

print(f"\n[2b] ORDER-{V28_ORDER} ONE-FACE PHYSICAL SMOKE — NO LOWER-ORDER COMPARATOR")
_v28_one_face = frozenset((V23C_ROOT,))
V28_ONE_FACE = _v28_cluster_coefficients(_v28_one_face, V28_ORDER)
_v28_one_face_finite = bool(np.all(np.isfinite(np.asarray(V28_ONE_FACE["coef"]))))
_v28_one_face_ok = (
    _v28_one_face_finite
    and V28_ONE_FACE["one_herm"] < V28_HERM_TOL
    and V28_ONE_FACE["vac_herm"] < V28_HERM_TOL
    and V28_ONE_FACE["sw_offdiag"] < 2e-8
    and len(V28_ONE_FACE["coef"]) >= V28_ORDER + 1
)
v28_gate(
    f"order-{V28_ORDER} one-face physical Gram/SW smoke passes",
    _v28_one_face_ok,
    f"layers={V28_ONE_FACE['one_layers']}/{V28_ONE_FACE['vac_layers']}; "
    f"SWoff={V28_ONE_FACE['sw_offdiag']:.3e}; finite={_v28_one_face_finite}",
)
V28_ONE_FACE_PREFIX_ERROR = None

'''
    source = source[:regression_start] + current_order_block + source[regression_end:]

    # Reuse the M5 one-face smoke as its production shape; do not calculate it twice.
    source = _replace_once(
        source,
        '    cache = _v28_load()\n    representatives = {}\n',
        r'''    cache = _v28_load()
    _one_key = _v24c_shape_key(frozenset((V23C_ROOT,)))
    if _one_key in V28_SHAPE_KEYS and _one_key not in cache:
        cache[_one_key] = V28_ONE_FACE
        _v28_save(cache)
        print("  seeded requested-order one-face smoke into production checkpoint")
    representatives = {}
''',
        "production smoke reuse",
    )

    # Remove the entire O4 support-corpus regression and generate the requested order directly.
    o4_start = source.index('    print("\\n[3] GENERIC SUPPORT-CENSUS REGRESSION AT ORDER FOUR")')
    order_start = source.index('    print(f"\\n[4] ORDER-{V28_ORDER} SUPPORT CENSUS")', o4_start)
    direct_order_block = r'''    print(f"\n[3] DIRECT ORDER-{V28_ORDER} HISTORY GENERATION — NO ORDER-4 CENSUS")
    if _v28_census_cache is None:
        _v28_histories = _v28_history_levels(V28_SUPPORT_HALF_DEPTH, V23C_POL)
    else:
        _v28_histories = None
        print(f"  loaded order-{V28_ORDER} census checkpoint")
    V28_O4_SUPPORTS = set()
    V28_O4_STATS = {"skipped": True, "reason": "requested-order-only runner"}

'''
    source = source[:o4_start] + direct_order_block + source[order_start:]

    # Remove every hard-coded lower-order target, including the M4 target gate.
    lower_start = source.index('def _v28_lower_targets():')
    production_start = source.index('\n\ndef _v28_production():', lower_start)
    source = source[:lower_start] + 'def _v28_lower_targets():\n    return {}\n' + source[production_start:]

    # Explicit runtime banner.
    source = _replace_once(
        source,
        'print("watchdog                     : NONE")\n',
        'print("watchdog                     : NONE")\n'
        'print("order-4 execution            : DISABLED")\n'
        'print("order-4 support census       : REMOVED")\n'
        'print("lower-order target gates     : REMOVED")\n'
        'print("support-census start order   :", V28_ORDER)\n',
        "runtime banner",
    )

    required_absent = (
        '[2b] ONE-FACE PHYSICAL PREFIX REGRESSION',
        'GENERIC SUPPORT-CENSUS REGRESSION AT ORDER FOUR',
        'through O4--O7',
        'completed v10a.26 one-face comparator is unavailable',
        '_v23c_fit_cluster(',
        '"_v10a3_h0_state_inner"',
        '"_v26_sw_blocks"',
        'for order in (4, 5, 6, 7):',
        'V28_ORDER = int(os.environ.get("V28_ORDER", "4"))',
        'if V28_ORDER not in (4, 5, 6, 7):',
    )
    leftovers = [token for token in required_absent if token in source]
    if leftovers:
        raise RuntimeError(f"Requested-order-only patch is incomplete; leftovers={leftovers}")
    if source == original:
        raise RuntimeError("v10a.28 patch made no changes")

    compile(source, "<v10a31-requested-order-only-engine>", "exec")
    return source


V28_PATCHED_SOURCE = patch_v28_m5_only(V28_ORIGINAL_SOURCE)
V28_PATCHED_PATH = SOURCE_DIR / f"Hodge_v10a31_M5_ONLY_engine_{V28_SHA256[:12]}.py"
V28_PATCHED_PATH.write_text(V28_PATCHED_SOURCE, encoding="utf-8")
V28_PATCHED_SHA256 = sha256_file(V28_PATCHED_PATH)

print("Patched engine:", V28_PATCHED_PATH)
print("Patched SHA-256:", V28_PATCHED_SHA256)
print("Order-4 runtime path: DISABLED")
print("Order-4 physical comparator: REMOVED")
print("Order-4 support census: REMOVED")
print("M4/lower-order target gates: REMOVED")
print("Requested-order targets: ABSENT")


## Start here: one blind M5 invocation

This is the first scientific run. The requested-order firewall, M5 support census, and rooted M5 production occur in the same execution of the patched engine. The M5 one-face smoke is checkpointed and reused rather than computed twice.


In [ ]:
def configure_order(order: int, *, max_new_shapes: int, time_budget_minutes: float):
    order = int(order)
    if order not in (5, 6, 7):
        raise ValueError("This runner permits orders 5, 6, and 7 only")

    cap = 7 if order == 5 else 9
    order_dir = WORKDIR / f"m{order}"
    order_dir.mkdir(parents=True, exist_ok=True)
    settings = {
        "V28_ORDER": str(order),
        "V28_MODE": "production",
        "V28_HAAR_CAP": str(cap),
        "V28_RUN_CENSUS": "1",
        "V28_GPU": "1",
        "V28_GPU_SW_MIN_DIM": str(GPU_SW_MIN_DIM),
        "V28_HERMITICITY_AUDIT_PAIRS": str(HERM_AUDIT_PAIRS),
        "V28_DUPLICATE_CHECKS": str(DUPLICATE_CHECKS),
        "V28_HEARTBEAT": str(HEARTBEAT_SECONDS),
        "V28_RESUME": "1",
        "V28_ALLOW_REFERENCE_REBUILD": "0",
        "V28_PRODUCTION_CONFIRM": f"YES_ORDER_{order}",
        "V28_MAX_NEW_SHAPES": str(int(max_new_shapes)),
        "V28_TIME_BUDGET_MINUTES": str(float(time_budget_minutes)),
        "V28_CHECKPOINT": str(order_dir / "requested_order_shapes.pkl"),
        "V28_CENSUS_CHECKPOINT": str(order_dir / "requested_order_census.pkl"),
    }
    os.environ.update(settings)
    return order_dir, settings


def execute_order(order: int, *, max_new_shapes: int, time_budget_minutes: float):
    order = int(order)
    order_dir, settings = configure_order(
        order,
        max_new_shapes=max_new_shapes,
        time_budget_minutes=time_budget_minutes,
    )

    # Remove prior execution products while preserving the source/patch objects.
    for global_name in tuple(globals()):
        if global_name.startswith("V28_") and global_name not in {
            "V28_PATH", "V28_SHA256", "V28_ORIGINAL_SOURCE",
            "V28_PATCHED_SOURCE", "V28_PATCHED_PATH", "V28_PATCHED_SHA256",
        }:
            globals().pop(global_name, None)

    print("=" * 108)
    print(f"BEGIN BLIND M{order} — REQUESTED-ORDER-ONLY ENGINE")
    print("NO ORDER-4 EXECUTION, CENSUS, COMPARATOR, OR TARGET GATE")
    print("=" * 108)
    started = time.time()
    exec(compile(V28_PATCHED_SOURCE, str(V28_PATCHED_PATH), "exec"), globals())
    elapsed = time.time() - started
    result = globals().get("V28_RESULT")
    if result is None:
        raise RuntimeError("Patched engine did not return V28_RESULT")
    print(f"Order-{order} invocation elapsed: {elapsed / 3600:.3f} h")
    print("Complete:", result.get("complete", True))
    return result, elapsed, order_dir, settings


V28_RESULT, ELAPSED_M5, M5_DIR, M5_SETTINGS = execute_order(
    5,
    max_new_shapes=M5_MAX_NEW_SHAPES,
    time_budget_minutes=M5_TIME_BUDGET_MINUTES,
)

if not V28_RESULT.get("complete", True):
    print("Atomic checkpoint saved. Rerun this cell to continue completed M5 shapes.")


In [ ]:
# Freeze the blind M5 result only after all M5 shapes complete.
def jsonable(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(item) for item in value]
    return value


M5_SUMMARY = M5_DIR / "blind_m5_summary.json"
if V28_RESULT.get("complete", True):
    coefficients = np.asarray(V28_RESULT["coefficients"], dtype=float)
    payload = {
        "schema": "hodge-v10a31-m5-only-blind-freeze-v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "order": 5,
        "blind": True,
        "external_requested_order_target_loaded": False,
        "v10a26_notebook_executed": False,
        "v10a26_definitions_only_bootstrap": True,
        "order4_runtime_permitted": False,
        "m4_physical_comparator_present": False,
        "o4_support_census_present": False,
        "m4_target_gate_present": False,
        "separate_firewall_invocation": False,
        "m5_engine_invocations_this_cell": 1,
        "coefficient_vector_m0_to_m5": coefficients.tolist(),
        "m5": float(coefficients[5]),
        "v10a26_source_sha256": V26_SHA256,
        "v10a26_bootstrap_sha256": sha256_file(BOOTSTRAP_PATH),
        "v10a28_source_sha256": V28_SHA256,
        "requested_order_engine_sha256": V28_PATCHED_SHA256,
        "v28_schema": V28_RESULT.get("schema"),
        "v28_signature": V28_RESULT.get("signature"),
        "concrete_clusters": int(V28_RESULT.get("concrete_clusters", 0)),
        "shape_classes": int(V28_RESULT.get("shapes", 0)),
        "haar_cap": int(V28_RESULT.get("haar_cap", 0)),
        "krylov_depth": int(V28_RESULT.get("krylov_depth", 0)),
        "gates": [
            {"name": name, "passed": bool(passed), "detail": detail}
            for name, passed, detail in globals().get("V28_GATES", [])
        ],
        "gpu": GPU_NAME,
        "elapsed_seconds_last_invocation": float(ELAPSED_M5),
        "checkpoint": M5_SETTINGS["V28_CHECKPOINT"],
        "census_checkpoint": M5_SETTINGS["V28_CENSUS_CHECKPOINT"],
    }
    M5_SUMMARY.write_text(json.dumps(jsonable(payload), indent=2), encoding="utf-8")
    print("FROZEN BLIND m5:", repr(float(coefficients[5])))
    print("Summary:", M5_SUMMARY)
    print("Summary SHA-256:", sha256_file(M5_SUMMARY))
else:
    print("M5 is incomplete; no coefficient summary has been frozen.")


## M6 and M7

The same requested-order-only engine is used. There is no order-4 path and no separate firewall run. Each order has independent checkpoints and a separate blind summary.


In [ ]:
def freeze_higher_order(order: int, result: dict, elapsed: float, order_dir: Path, settings: dict):
    coefficients = np.asarray(result["coefficients"], dtype=float)
    summary = order_dir / f"blind_m{order}_summary.json"
    lower_self_consistency = {}
    for lower_order in range(5, order):
        prior = WORKDIR / f"m{lower_order}" / f"blind_m{lower_order}_summary.json"
        if prior.exists():
            old = json.loads(prior.read_text(encoding="utf-8"))
            lower_self_consistency[f"m{lower_order}"] = {
                "current": float(coefficients[lower_order]),
                "frozen": float(old[f"m{lower_order}"]),
                "abs_error": abs(float(coefficients[lower_order]) - float(old[f"m{lower_order}"])),
            }

    payload = {
        "schema": f"hodge-v10a31-m5-only-blind-m{order}-freeze-v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "order": order,
        "blind": True,
        "external_requested_order_target_loaded": False,
        "order4_runtime_permitted": False,
        "m4_physical_comparator_present": False,
        "o4_support_census_present": False,
        "m4_target_gate_present": False,
        f"m{order}": float(coefficients[order]),
        "coefficient_vector": coefficients.tolist(),
        "lower_order_self_consistency": lower_self_consistency,
        "v10a26_source_sha256": V26_SHA256,
        "v10a26_bootstrap_sha256": sha256_file(BOOTSTRAP_PATH),
        "v10a28_source_sha256": V28_SHA256,
        "requested_order_engine_sha256": V28_PATCHED_SHA256,
        "v28_schema": result.get("schema"),
        "v28_signature": result.get("signature"),
        "gates": [
            {"name": name, "passed": bool(passed), "detail": detail}
            for name, passed, detail in globals().get("V28_GATES", [])
        ],
        "gpu": GPU_NAME,
        "elapsed_seconds_last_invocation": float(elapsed),
        "checkpoint": settings["V28_CHECKPOINT"],
        "census_checkpoint": settings["V28_CENSUS_CHECKPOINT"],
    }
    summary.write_text(json.dumps(jsonable(payload), indent=2), encoding="utf-8")
    print(f"FROZEN BLIND m{order}:", repr(float(coefficients[order])))
    print("Summary:", summary)
    print("Summary SHA-256:", sha256_file(summary))
    return summary


def run_higher_order(
    order: int,
    *,
    max_new_shapes: int = HIGHER_ORDER_MAX_NEW_SHAPES,
    time_budget_minutes: float = HIGHER_ORDER_TIME_BUDGET_MINUTES,
):
    order = int(order)
    if order not in (6, 7):
        raise ValueError("order must be 6 or 7")

    result, elapsed, order_dir, settings = execute_order(
        order,
        max_new_shapes=max_new_shapes,
        time_budget_minutes=time_budget_minutes,
    )
    if not result.get("complete", True):
        print(f"M{order} checkpoint saved; rerun run_higher_order({order}) to continue.")
        return result
    freeze_higher_order(order, result, elapsed, order_dir, settings)
    return result


# After M5 completes, run as needed:
# run_higher_order(6)
# run_higher_order(7)


In [ ]:
# Compact provenance bundle. Large pickle checkpoints remain in Drive.
bundle = WORKDIR / "blind_results_m5_only_compact.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(WORKDIR.rglob("*")):
        if not path.is_file() or path.suffix == ".pkl" or path == bundle:
            continue
        archive.write(path, path.relative_to(WORKDIR))
    archive.write(V26_PATH, Path("original_sources") / V26_PATH.name)
    archive.write(V28_PATH, Path("original_sources") / V28_PATH.name)
    archive.write(BOOTSTRAP_PATH, Path("generated_sources") / BOOTSTRAP_PATH.name)
    archive.write(V28_PATCHED_PATH, Path("generated_sources") / V28_PATCHED_PATH.name)

print("Compact bundle:", bundle)
print("SHA-256:", sha256_file(bundle))


## Return after the run

For M5, return `blind_m5_summary.json`, the final gate summary, and the output around `ROOTED INCIDENCE TRANSFORM`. A correct startup explicitly reports that order-4 execution, the order-4 support census, the physical comparator, and lower-order target gates are removed.
